In [85]:
# the data is Loaded here

In [147]:
import pandas as pd
import numpy as np

In [87]:
loaded_df=pd.read_csv("spam_ham_india.csv")

In [88]:
df=loaded_df

In [89]:
print('#'*128)

################################################################################################################################


In [90]:
# data is coverted to lower case, we have two ways

In [91]:
df['Msg']=df['Msg'].str.lower()

In [92]:
# for i,row in df.iterrows():
#     df.at[i,'Msg']=str(row.Msg).lower()

In [93]:
df_lower=df

In [94]:
print('#'*128)

################################################################################################################################


In [95]:
# data is cleaned here

In [96]:
from bs4 import BeautifulSoup as bs
import re
import string

In [97]:
def clean_text(text):
    text = bs(text, "html.parser").get_text()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text



In [98]:
df_lower['Msg']=df_lower['Msg'].apply(lambda x:clean_text(str(x)))

In [99]:
df_clean=df_lower

In [100]:
print('#'*128)

################################################################################################################################


In [101]:
# data is tokenized here

In [102]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [103]:
tokenizer=Tokenizer()

In [104]:
tokenizer.fit_on_texts(df_clean['Msg'])

In [105]:
tokenizer.word_index.get(20)

In [106]:
df_clean['Msg']=tokenizer.texts_to_sequences(df_clean['Msg'])

In [107]:
df_tokenized=df_clean

In [108]:
print('#'*128)

################################################################################################################################


In [109]:
# data is padded here

In [110]:
longest_row=max(len(row['Msg']) for i,row in df_tokenized.iterrows())

In [111]:
input_list=df_tokenized['Msg']

In [112]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input=pad_sequences(input_list,maxlen=longest_row,padding='pre')

In [113]:
padded_input

array([[   0,    0,    0, ...,   17,  173, 2383],
       [   0,    0,    0, ...,  212,    5,  322],
       [   0,    0,    0, ..., 1268,  890,   18],
       ...,
       [   0,    0,    0, ...,  110,  510, 4576],
       [   0,    0,    0, ...,   17, 4577,   40],
       [   0,    0,    0, ..., 1036,  510, 4578]])

In [114]:
print('#'*128)

################################################################################################################################


In [115]:
# readying input and output for the models

In [116]:
'''we have an input in form of padded squence now we need to make an output for spam and ham '''

'we have an input in form of padded squence now we need to make an output fot spam and ham '

In [125]:
ot=[ 1 if(x=='spam') else 0 for x in df_tokenized['Label']]

In [148]:
x=padded_input
y=np.array(ot)

In [135]:
x.shape

(2267, 153)

In [137]:
len(y)

2267

In [138]:
len(tokenizer.word_index)

4578

In [ ]:
# **now the possible problem can be input is an np  array, can be dealt with later**

In [123]:
print('#'*128)

################################################################################################################################


In [ ]:
# now we have to work with layers and models of lstm

In [ ]:
'''
three layers will be there in the model 
one wil be input layer 
one will be lstm layer
one will be a dense output layer with one output node to get 1, 0
'''

In [177]:
pip install sklearn

  Created wheel for sklearn: filename=sklearn-0.0-py2.py3-none-any.whl size=1324 sha256=6c278f5f70149b3f1091bd8664f873e8dea8ba9365ce41e32c3b581c07705807
  Stored in directory: c:\users\abhin\appdata\local\pip\cache\wheels\22\0b\40\fd3f795caaa1fb4c6cb738bc1f56100be1e57da95849bfc897
Successfully built sklearn
Note: you may need to restart the kernel to use updated packages.


    ERROR: Command errored out with exit status 1:
     command: 'C:\Users\abhin\Desktop\Spam\.venv\scripts\python.exe' -c 'import io, os, sys, setuptools, tokenize; sys.argv[0] = '"'"'C:\\Users\\abhin\\AppData\\Local\\Temp\\pip-install-vh834skk\\sklearn_5e408767e2f7417a98229856a9f8421b\\setup.py'"'"'; __file__='"'"'C:\\Users\\abhin\\AppData\\Local\\Temp\\pip-install-vh834skk\\sklearn_5e408767e2f7417a98229856a9f8421b\\setup.py'"'"';f = getattr(tokenize, '"'"'open'"'"', open)(__file__) if os.path.exists(__file__) else io.StringIO('"'"'from setuptools import setup; setup()'"'"');code = f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"'))' egg_info --egg-base 'C:\Users\abhin\AppData\Local\Temp\pip-pip-egg-info-733a6_lv'
         cwd: C:\Users\abhin\AppData\Local\Temp\pip-install-vh834skk\sklearn_5e408767e2f7417a98229856a9f8421b\
    Complete output (15 lines):
    The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
    rather than

In [179]:
from sklearn.model_selection import train_test_split

# Initial split: 80% for training+validation, 20% for final testing
X_train_val, X_test, y_train_val, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Second split: Divide the 80% into 75% training and 25% validation 
# (This results in 60% Train, 20% Val, 20% Test overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)


In [181]:
X_train_val.shape

(1813, 153)

In [164]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

In [183]:
model=Sequential()



model.add(Embedding(input_dim=4579,input_length=153,output_dim=32))

model.add(Dropout(0.2))

model.add(LSTM(32))

model.add(Dense(1,activation='sigmoid'))





model.compile( loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy'])

model.summary()
# ------------------ 6. TRAIN (WITH VALIDATION) ------------------
model.fit(
    X_train, y_train,
    epochs=15,
    validation_data=(X_val, y_val)
)

Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_8 (Embedding)     (None, 153, 32)           146528    
                                                                 
 dropout_4 (Dropout)         (None, 153, 32)           0         
                                                                 
 lstm_7 (LSTM)               (None, 32)                8320      
                                                                 
 dense_7 (Dense)             (None, 1)                 33        
                                                                 
Total params: 154881 (605.00 KB)
Trainable params: 154881 (605.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/15
43/43 [==============================] - 5s 86ms/step - loss: 0.5253 - accuracy: 0.7167 - val_loss: 0.2890 - val_accuracy: 0.9427
Epoch

In [184]:
# ------------------ 7. FINAL TEST ------------------
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_acc)

15/15 [==============================] - 0s 14ms/step - loss: 0.0426 - accuracy: 0.9890
Test Accuracy: 0.9889867901802063


In [185]:
y_pred=model.predict(X_test)

15/15 [==============================] - 0s 18ms/step


In [188]:
y_pred=np.array([1 if(x>0.7) else 0 for x in y_pred])

In [189]:
from sklearn.metrics import confusion_matrix

# y_true = actual labels, y_pred = model's predicted labels
y_true = y_test


cm = confusion_matrix(y_true, y_pred)
print(cm)


[[293   4]
 [  3 154]]


In [190]:
import pickle

In [192]:
with open("model.pkl", "wb") as file:
    pickle.dump(model,file)

In [194]:
#/////////////////////////////////////////////////////////Saved the file ///////////////////////////////////////////////////////////////////////////////#